# ★ AI를 활용한 정보보안 프로그램 만들어보기
## 암복호화 프로그램




*   역할 : 너는 보안분야 전문 프로그래머야.
*   목표 : 시저암호를 이용한 간단한 데이터 암복호화 프로그램을 만들어줘.
*   입력 :
1) 사용자로부터 모드를 선택받음 ( 1. 암호화 2. 복호화 3. 종료)
2) 모드에 따라서 평문 또는 암호문 그리고 키를 입력받음
3) 사용자가 종료를 선택하기 전까지 프로그램 계속 진행.
*   출력 : 처리된 암호문 또는 평문 출력



In [ ]:
def caesar_cipher(text, shift, mode):
    result = ""
    # 복호화일 경우 shift를 음수로 변환
    if mode == '2':
        shift = -shift

    for char in text:
        if char.isupper():
            result += chr((ord(char) + shift - 65) % 26 + 65)
        elif char.islower():
            result += chr((ord(char) + shift - 97) % 26 + 97)
        else:
            result += char
    return result

# 메인 루프: 사용자가 3번(종료)을 누르기 전까지 무한 반복
while True:
    print("\n--- 🛡️ 시저 암호 보안 시스템 ---")
    print("1. 암호화(Encrypt)  2. 복호화(Decrypt)  3. 종료(Exit)")

    choice = input("작업할 모드를 선택하세요 (1/2/3): ")

    if choice == '3':
        print("보안 프로그램을 종료합니다. 안전한 하루 되세요!")
        break

    if choice in ['1', '2']:
        target_text = input("메시지를 입력하세요: ")

        # 숫자 키 입력 예외 처리 (숫자가 아닌 경우 대비)
        try:
            key = int(input("암호 키(숫자 1~25)를 입력하세요: "))
        except ValueError:
            print("❌ 오류: 키는 숫자여야 합니다. 처음으로 돌아갑니다.")
            continue

        processed_text = caesar_cipher(target_text, key, choice)

        mode_name = "암호화" if choice == '1' else "복호화"
        print("-" * 30)
        print(f"✅ 결과 ({mode_name}): {processed_text}")
        print("-" * 30)
    else:
        print("❌ 잘못된 선택입니다. 1, 2, 3 중에서 골라주세요.")

# ★ AI 보안 실습: 스테가노그래피
## 이미지에 비밀 메시지 숨기기

In [ ]:
import cv2
import numpy as np
from google.colab import files

def encode_message():
    print("1. 이미지를 업로드하세요.")
    uploaded = files.upload()
    if not uploaded: return
    img_path = list(uploaded.keys())[0]
    image = cv2.imread(img_path)

    secret_data = input("2. 숨길 비밀 메시지를 입력하세요: ")
    secret_data += "####" # 메시지의 끝을 알리는 표식

    # 메시지를 이진수로 변환
    binary_secret = ''.join(format(ord(i), '08b') for i in secret_data)

    data_idx = 0
    data_len = len(binary_secret)

    # 픽셀을 하나씩 돌며 하위 1비트에 데이터 저장
    for values in image:
        for pixel in values:
            for i in range(3): # B, G, R 채널
                if data_idx < data_len:
                    # 픽셀의 마지막 비트를 0으로 만들고 메시지 비트를 더함
                    # ~1 대신 np.uint8(254)를 사용하여 음수 OverflowError 방지
                    pixel[i] = (pixel[i] & np.uint8(254)) | int(binary_secret[data_idx])
                    data_idx += 1
        if data_idx >= data_len: break

    cv2.imwrite("stego_image.png", image)
    print("\n✅ 성공! 'stego_image.png' 파일이 생성되었습니다.")
    files.download("stego_image.png")

encode_message()

# ★ AI 보안 실습: 스테가노그래피
## 이미지에 숨겨진 비밀 메시지 찾기

In [ ]:
import cv2
import numpy as np
from google.colab import files

def decode_message():
    print("메시지를 추출할 이미지를 업로드하세요.")
    uploaded = files.upload()
    if not uploaded: return
    img_path = list(uploaded.keys())[0]
    image = cv2.imread(img_path)

    binary_data = ""
    for values in image:
        for pixel in values:
            for i in range(3):
                binary_data += str(pixel[i] & 1)

    # 8비트씩 잘라서 문자로 변환
    all_bytes = [binary_data[i: i+8] for i in range(0, len(binary_data), 8)]
    decoded_data = ""
    for byte in all_bytes:
        decoded_data += chr(int(byte, 2))
        if decoded_data[-4:] == "####": # 끝 표식을 만나면 멈춤
            print("\n🔍 추출된 메시지:", decoded_data[:-4])
            return

decode_message()

# ★ AI 시대의 개인정보 비식별화
## 실시간 얼굴 모자이크 처리

In [ ]:
# 1. 필요한 라이브러리 가져오기
import cv2
import matplotlib.pyplot as plt
from google.colab import files
import numpy as np

# 2. 실습할 이미지 업로드하기
print("실습할 이미지 파일을 업로드하세요.")
uploaded = files.upload()

for fn in uploaded.keys():
  # 이미지 읽어오기
  img = cv2.imread(fn)
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

  # 3. 얼굴 찾기 도구(Haar Cascade) 불러오기
  # OpenCV에서 기본 제공하는 얼굴 인식 모델입니다.
  face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

  # 4. 이미지에서 얼굴 탐지하기
  faces = face_cascade.detectMultiScale(gray, 1.3, 5)

  # 5. 찾은 얼굴 부위만 모자이크 처리하기
  for (x, y, w, h) in faces:
      # 얼굴 영역 추출
      face_roi = img[y:y+h, x:x+w]

      # 모자이크 처리 (축소 후 확대)
      # 15x15 크기로 줄였다가 원래 크기로 키워 픽셀을 뭉개뜨립니다.
      shrink = cv2.resize(face_roi, (30, 30), interpolation=cv2.INTER_NEAREST)
      mosaic = cv2.resize(shrink, (w, h), interpolation=cv2.INTER_NEAREST)

      # 원본 이미지에 모자이크 입히기
      img[y:y+h, x:x+w] = mosaic

      # 얼굴 주위에 사각형 표시 (선택 사항)
      cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)

  # 6. 결과 화면에 출력하기
  img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  plt.figure(figsize=(10, 10))
  plt.imshow(img_rgb)
  plt.axis('off')
  plt.show()
  print(f"총 {len(faces)}개의 얼굴을 찾아 모자이크 처리했습니다!")